In [53]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

customer_file = list(uploaded.keys())[0]

df_customer = pd.read_csv(customer_file)

print("Customer dataset loaded successfully!")
print("Shape:", df_customer.shape)

Saving customer_data_collection.csv to customer_data_collection (1).csv
Customer dataset loaded successfully!
Shape: (10000, 11)


In [2]:
print("\n--- COLUMNS ---")
print(df_customer.columns.tolist())

print("\n--- DATA TYPES ---")
print(df_customer.dtypes)

print("\n--- MISSING VALUES ---")
print(df_customer.isnull().sum())

print("\n--- FIRST 5 ROWS ---")
display(df_customer.head())


--- COLUMNS ---
['Customer_ID', 'Age', 'Gender', 'Location', 'Browsing_History', 'Purchase_History', 'Customer_Segment', 'Avg_Order_Value', 'Holiday', 'Season', 'Unnamed: 10']

--- DATA TYPES ---
Customer_ID          object
Age                   int64
Gender               object
Location             object
Browsing_History     object
Purchase_History     object
Customer_Segment     object
Avg_Order_Value     float64
Holiday              object
Season               object
Unnamed: 10         float64
dtype: object

--- MISSING VALUES ---
Customer_ID             0
Age                     0
Gender                  0
Location                0
Browsing_History        0
Purchase_History        0
Customer_Segment        0
Avg_Order_Value         0
Holiday                 0
Season                  0
Unnamed: 10         10000
dtype: int64

--- FIRST 5 ROWS ---


,Customer_ID,Age,Gender,Location,Browsing_History,Purchase_History,Customer_Segment,Avg_Order_Value,Holiday,Season,Unnamed: 10
0,C1000,28,Female,Chennai,"['Books', 'Fashion']","['Biography', 'Jeans']",New Visitor,4806.99,No,Winter,NaN
1,C1001,27,Male,Delhi,"['Books', 'Fitness', 'Fashion']","['Biography', 'Resistance Bands', 'T-shirt']",Occasional Shopper,795.03,Yes,Autumn,NaN
2,C1002,34,Other,Chennai,['Electronics'],['Smartphone'],Occasional Shopper,1742.45,Yes,Summer,NaN
3,C1003,23,Male,Bangalore,['Home Decor'],['Wall Art'],Frequent Buyer,2023.16,No,Autumn,NaN
4,C1004,24,Other,Kolkata,"['Fashion', 'Home Decor']","['Shoes', 'Lamp']",Frequent Buyer,794.76,No,Winter,NaN


In [3]:
import ast
import pandas as pd

# Convert the string representation of lists into actual Python lists
df_customer["Browsing_List"] = df_customer["Browsing_History"].apply(
    ast.literal_eval
)

df_customer["Purchase_List"] = df_customer["Purchase_History"].apply(
    ast.literal_eval
)

print("Customer records:", len(df_customer))

print("\nTotal browsing interactions:",
      df_customer["Browsing_List"].apply(len).sum())

print("Total purchase interactions:",
      df_customer["Purchase_List"].apply(len).sum())

print("\nAverage browsed items per customer:",
      df_customer["Browsing_List"].apply(len).mean())

print("Average purchased items per customer:",
      df_customer["Purchase_List"].apply(len).mean())

Customer records: 10000

Total browsing interactions: 19967
Total purchase interactions: 19967

Average browsed items per customer: 1.9967
Average purchased items per customer: 1.9967


In [4]:
# Flatten purchase history
purchase_items = [
    item
    for items in df_customer["Purchase_List"]
    for item in items
]

print("Unique purchased products/items:",
      len(set(purchase_items)))

print("\nMost frequently purchased items:")
print(
    pd.Series(purchase_items)
    .value_counts()
    .head(20)
)

Unique purchased products/items: 24

Most frequently purchased items:
Moisturizer         882
Curtains            876
T-shirt             875
Smartphone          863
Headphones          856
Lipstick            855
Jeans               852
Cushions            843
Wall Art            841
Biography           836
Resistance Bands    831
Comics              829
Dumbbells           827
Treadmill           827
Yoga Mat            824
Laptop              824
Foundation          823
Smartwatch          823
Jacket              818
Non-fiction         805
Name: count, dtype: int64


In [5]:
# Flatten browsing history
browse_items = [
    item
    for items in df_customer["Browsing_List"]
    for item in items
]

print("Unique browsed items:",
      len(set(browse_items)))

print("\nMost frequently browsed items:")
print(
    pd.Series(browse_items)
    .value_counts()
    .head(20)
)

Unique browsed items: 6

Most frequently browsed items:
Electronics    3366
Beauty         3348
Home Decor     3345
Fashion        3333
Fitness        3309
Books          3266
Name: count, dtype: int64


In [6]:
purchase_counts = (
    pd.Series(purchase_items)
    .value_counts()
    .sort_values(ascending=False)
)

print(purchase_counts)

Moisturizer         882
Curtains            876
T-shirt             875
Smartphone          863
Headphones          856
Lipstick            855
Jeans               852
Cushions            843
Wall Art            841
Biography           836
Resistance Bands    831
Comics              829
Dumbbells           827
Treadmill           827
Yoga Mat            824
Laptop              824
Foundation          823
Smartwatch          823
Jacket              818
Non-fiction         805
Fiction             796
Shoes               788
Perfume             788
Lamp                785
Name: count, dtype: int64


In [7]:
print("\nNumber of customers with each purchase count:")

purchase_count_per_customer = (
    df_customer["Purchase_List"]
    .apply(len)
    .value_counts()
    .sort_index()
)

print(purchase_count_per_customer)


Number of customers with each purchase count:
Purchase_List
1    3369
2    3295
3    3336
Name: count, dtype: int64


In [8]:
print("\nCustomers with 1 purchase item:")
print((df_customer["Purchase_List"].apply(len) == 1).sum())

print("\nCustomers with 2 purchase items:")
print((df_customer["Purchase_List"].apply(len) == 2).sum())

print("\nCustomers with 3 purchase items:")
print((df_customer["Purchase_List"].apply(len) == 3).sum())


Customers with 1 purchase item:
3369

Customers with 2 purchase items:
3295

Customers with 3 purchase items:
3336


In [9]:
import pandas as pd

purchase_interactions = (
    df_customer[["Customer_ID", "Purchase_List"]]
    .explode("Purchase_List")
    .rename(columns={"Purchase_List": "Product"})
)

purchase_interactions["Purchase"] = 1

print(purchase_interactions.head())
print("\nShape:", purchase_interactions.shape)

  Customer_ID           Product  Purchase
0       C1000         Biography         1
0       C1000             Jeans         1
1       C1001         Biography         1
1       C1001  Resistance Bands         1
1       C1001           T-shirt         1

Shape: (19967, 3)


In [10]:
user_item_matrix = pd.crosstab(
    purchase_interactions["Customer_ID"],
    purchase_interactions["Product"]
)

print("Matrix shape:", user_item_matrix.shape)
display(user_item_matrix.head())

Matrix shape: (10000, 24)


Product,Biography,Comics,Curtains,Cushions,Dumbbells,Fiction,Foundation,Headphones,Jacket,Jeans,...,Non-fiction,Perfume,Resistance Bands,Shoes,Smartphone,Smartwatch,T-shirt,Treadmill,Wall Art,Yoga Mat
Customer_ID,,,,,,,,,,,,,,,,,,,,,
C1000,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
C10000,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
C10001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
C10002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
C10003,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(user_item_matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

display(item_similarity_df.round(3))

Product,Biography,Comics,Curtains,Cushions,Dumbbells,Fiction,Foundation,Headphones,Jacket,Jeans,...,Non-fiction,Perfume,Resistance Bands,Shoes,Smartphone,Smartwatch,T-shirt,Treadmill,Wall Art,Yoga Mat
Product,,,,,,,,,,,,,,,,,,,,,
Biography,1.000,0.000,0.071,0.055,0.066,0.000,0.053,0.066,0.057,0.060,...,0.000,0.075,0.079,0.060,0.064,0.072,0.065,0.067,0.069,0.073
Comics,0.000,1.000,0.061,0.073,0.053,0.000,0.075,0.059,0.075,0.050,...,0.000,0.063,0.059,0.079,0.067,0.079,0.072,0.064,0.060,0.073
Curtains,0.071,0.061,1.000,0.000,0.046,0.068,0.068,0.072,0.074,0.067,...,0.058,0.072,0.063,0.078,0.078,0.077,0.077,0.060,0.000,0.068
Cushions,0.055,0.073,0.000,1.000,0.062,0.059,0.066,0.048,0.069,0.081,...,0.066,0.074,0.059,0.065,0.068,0.065,0.078,0.087,0.000,0.056
Dumbbells,0.066,0.053,0.046,0.062,1.000,0.070,0.069,0.064,0.066,0.081,...,0.071,0.069,0.000,0.052,0.072,0.065,0.076,0.000,0.077,0.000
Fiction,0.000,0.000,0.068,0.059,0.070,1.000,0.067,0.076,0.068,0.085,...,0.000,0.061,0.064,0.064,0.049,0.073,0.071,0.059,0.055,0.083
Foundation,0.053,0.075,0.068,0.066,0.069,0.067,1.000,0.066,0.069,0.063,...,0.066,0.000,0.060,0.075,0.058,0.041,0.057,0.065,0.066,0.063
Headphones,0.066,0.059,0.072,0.048,0.064,0.076,0.066,1.000,0.075,0.059,...,0.060,0.072,0.047,0.071,0.000,0.000,0.065,0.074,0.059,0.076
Jacket,0.057,0.075,0.074,0.069,0.066,0.068,0.069,0.075,1.000,0.000,...,0.060,0.062,0.057,0.000,0.073,0.063,0.000,0.073,0.059,0.063


In [12]:
def recommend_products(customer_id, n=5):

    purchased = user_item_matrix.loc[customer_id]

    purchased_products = purchased[purchased > 0].index.tolist()

    scores = item_similarity_df[purchased_products].sum(axis=1)

    # Remove products already purchased
    scores = scores.drop(purchased_products, errors="ignore")

    recommendations = (
        scores
        .sort_values(ascending=False)
        .head(n)
    )

    return recommendations

In [13]:
customer_id = df_customer["Customer_ID"].iloc[0]

print("Customer:", customer_id)

print("\nPurchased products:")
print(
    df_customer.loc[
        df_customer["Customer_ID"] == customer_id,
        "Purchase_List"
    ].iloc[0]
)

print("\nRecommended products:")
print(recommend_products(customer_id, 5))

Customer: C1000

Purchased products:
['Biography', 'Jeans']

Recommended products:
Product
Dumbbells           0.147156
Resistance Bands    0.142172
Perfume             0.141060
Lamp                0.140118
Lipstick            0.138935
dtype: float64


In [14]:
print("Matrix shape:", user_item_matrix.shape)

Matrix shape: (10000, 24)


In [15]:
for product in ["Laptop", "Jeans", "Smartphone", "T-shirt"]:
    print(f"\nSimilar products to {product}:")
    print(
        item_similarity_df[product]
        .sort_values(ascending=False)
        .iloc[1:6]
    )


Similar products to Laptop:
Product
Non-fiction    0.078581
Wall Art       0.073277
T-shirt        0.073017
Cushions       0.071990
Foundation     0.071645
Name: Laptop, dtype: float64

Similar products to Jeans:
Product
Fiction      0.085001
Cushions     0.081417
Dumbbells    0.081010
Laptop       0.068029
Curtains     0.067136
Name: Jeans, dtype: float64

Similar products to Smartphone:
Product
Wall Art       0.084514
Moisturizer    0.083673
Lipstick       0.079163
Curtains       0.078208
Perfume        0.077609
Name: Smartphone, dtype: float64

Similar products to T-shirt:
Product
Cushions     0.078011
Lipstick     0.077462
Curtains     0.076528
Dumbbells    0.076411
Lamp         0.076015
Name: T-shirt, dtype: float64


In [16]:
import numpy as np

def recommend_from_items(purchased_products, n=5):

    scores = item_similarity_df[purchased_products].sum(axis=1)

    # Remove products already known to the model
    scores = scores.drop(
        labels=purchased_products,
        errors="ignore"
    )

    return scores.sort_values(ascending=False).head(n).index.tolist()

In [17]:
def evaluate_recommender(k=5):

    hits = 0
    total = 0

    precision_sum = 0
    recall_sum = 0

    for _, row in df_customer.iterrows():

        purchased = row["Purchase_List"]

        # Need at least 2 products
        if len(purchased) < 2:
            continue

        # Use the first products as training
        train_items = purchased[:-1]

        # Last product is the hidden test item
        test_item = purchased[-1]

        recommendations = recommend_from_items(
            train_items,
            n=k
        )

        # Hit Rate
        if test_item in recommendations:
            hits += 1

        # Precision@K
        relevant = len(set(recommendations) & {test_item})
        precision_sum += relevant / k

        # Recall@K
        recall_sum += relevant / 1

        total += 1

    hit_rate = hits / total
    precision = precision_sum / total
    recall = recall_sum / total

    return hit_rate, precision, recall

In [18]:
hit_rate, precision, recall = evaluate_recommender(k=5)

print(f"Hit Rate@5 : {hit_rate:.4f}")
print(f"Precision@5: {precision:.4f}")
print(f"Recall@5   : {recall:.4f}")

Hit Rate@5 : 0.3360
Precision@5: 0.0672
Recall@5   : 0.3360


In [19]:
product_popularity = (
    purchase_interactions["Product"]
    .value_counts()
)

print(product_popularity)

Product
Moisturizer         882
Curtains            876
T-shirt             875
Smartphone          863
Headphones          856
Lipstick            855
Jeans               852
Cushions            843
Wall Art            841
Biography           836
Resistance Bands    831
Comics              829
Dumbbells           827
Treadmill           827
Yoga Mat            824
Laptop              824
Foundation          823
Smartwatch          823
Jacket              818
Non-fiction         805
Fiction             796
Shoes               788
Perfume             788
Lamp                785
Name: count, dtype: int64


In [20]:
def popularity_recommend(customer_id, k=5):
    purchased = set(
        user_item_matrix.loc[customer_id]
        [user_item_matrix.loc[customer_id] > 0]
        .index
    )

    recommendations = [
        product
        for product in product_popularity.index
        if product not in purchased
    ]

    return recommendations[:k]

In [21]:
def evaluate_popularity(k=5):

    hits = 0
    total_precision = 0
    total_recall = 0
    evaluated_users = 0

    for customer_id, group in purchase_interactions.groupby("Customer_ID"):

        actual_items = group["Product"].tolist()

        # Need at least 2 purchases
        if len(actual_items) < 2:
            continue

        # Use first item as training item
        train_items = actual_items[:-1]
        test_item = actual_items[-1]

        recommendations = popularity_recommend(customer_id, k)

        if test_item in recommendations:
            hits += 1

        total_precision += (
            1 / k if test_item in recommendations else 0
        )

        total_recall += (
            1 if test_item in recommendations else 0
        )

        evaluated_users += 1

    hit_rate = hits / evaluated_users
    precision = total_precision / evaluated_users
    recall = total_recall / evaluated_users

    return hit_rate, precision, recall

In [22]:
pop_hit, pop_precision, pop_recall = evaluate_popularity(k=5)

print(f"Popularity Hit Rate@5 : {pop_hit:.4f}")
print(f"Popularity Precision@5: {pop_precision:.4f}")
print(f"Popularity Recall@5   : {pop_recall:.4f}")

Popularity Hit Rate@5 : 0.0000
Popularity Precision@5: 0.0000
Popularity Recall@5   : 0.0000


In [23]:
def evaluate_popularity_correct(k=5):

    hits = 0
    total_users = 0

    for customer_id, group in purchase_interactions.groupby("Customer_ID"):

        actual_items = group["Product"].tolist()

        # Only customers with at least 2 purchases
        if len(actual_items) < 2:
            continue

        # Hold out the last purchase
        test_item = actual_items[-1]
        train_items = set(actual_items[:-1])

        # Popular products
        recommendations = [
            product
            for product in product_popularity.index
            if product not in train_items
        ][:k]

        if test_item in recommendations:
            hits += 1

        total_users += 1

    hit_rate = hits / total_users

    return hit_rate

In [24]:
pop_hit_rate = evaluate_popularity_correct(k=5)

print(f"Corrected Popularity Hit Rate@5: {pop_hit_rate:.4f}")

Corrected Popularity Hit Rate@5: 0.2330


In [25]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def evaluate_item_cf(k=5):

    hits = 0
    total_users = 0

    for customer_id, group in purchase_interactions.groupby("Customer_ID"):

        actual_items = group["Product"].tolist()

        # Need at least 2 purchases
        if len(actual_items) < 2:
            continue

        # Hold out one purchase
        test_item = actual_items[-1]
        train_items = actual_items[:-1]

        # Calculate recommendation score from training items
        scores = pd.Series(
            0.0,
            index=item_similarity_df.index
        )

        for item in train_items:
            if item in item_similarity_df.columns:
                scores += item_similarity_df[item]

        # Remove products already seen during training
        scores = scores.drop(
            labels=train_items,
            errors="ignore"
        )

        recommendations = (
            scores
            .sort_values(ascending=False)
            .head(k)
            .index
            .tolist()
        )

        if test_item in recommendations:
            hits += 1

        total_users += 1

    hit_rate = hits / total_users

    return hit_rate

In [26]:
cf_hit_rate = evaluate_item_cf(k=5)

print(f"Corrected Item-CF Hit Rate@5: {cf_hit_rate:.4f}")

Corrected Item-CF Hit Rate@5: 0.3360


In [27]:
# Create browsing-category → purchased-product pairs

browsing_purchase_pairs = []

for _, row in df_customer.iterrows():

    browsing_categories = row["Browsing_List"]
    purchased_products = row["Purchase_List"]

    for category in browsing_categories:
        for product in purchased_products:
            browsing_purchase_pairs.append(
                (category, product)
            )

browsing_purchase_df = pd.DataFrame(
    browsing_purchase_pairs,
    columns=["Category", "Product"]
)

print("Browsing-Purchase pairs:", len(browsing_purchase_df))
display(browsing_purchase_df.head(20))

Browsing-Purchase pairs: 46573


,Category,Product
0,Books,Biography
1,Books,Jeans
2,Fashion,Biography
3,Fashion,Jeans
4,Books,Biography
5,Books,Resistance Bands
6,Books,T-shirt
7,Fitness,Biography
8,Fitness,Resistance Bands
9,Fitness,T-shirt


In [28]:
category_product_counts = pd.crosstab(
    browsing_purchase_df["Category"],
    browsing_purchase_df["Product"]
)

print("Category-Product matrix shape:",
      category_product_counts.shape)

display(category_product_counts)

Category-Product matrix shape: (6, 24)


Product,Biography,Comics,Curtains,Cushions,Dumbbells,Fiction,Foundation,Headphones,Jacket,Jeans,...,Non-fiction,Perfume,Resistance Bands,Shoes,Smartphone,Smartwatch,T-shirt,Treadmill,Wall Art,Yoga Mat
Category,,,,,,,,,,,,,,,,,,,,,
Beauty,223,219,242,242,228,216,823,219,214,221,...,235,788,220,210,254,209,230,229,225,200
Books,836,829,219,209,214,796,214,219,213,218,...,805,215,235,208,204,229,238,199,196,238
Electronics,221,228,245,213,227,215,197,856,230,200,...,211,232,197,220,863,823,232,224,242,225
Fashion,203,229,253,246,229,235,218,227,818,852,...,210,211,215,788,219,212,875,231,204,218
Fitness,238,206,202,221,827,224,213,220,213,228,...,218,204,831,212,215,217,240,827,237,824
Home Decor,227,218,876,843,207,200,226,212,205,235,...,204,218,217,209,247,229,252,238,841,231


In [29]:
category_product_probability = (
    category_product_counts
    .div(category_product_counts.sum(axis=1), axis=0)
)

display(category_product_probability.round(4))

Product,Biography,Comics,Curtains,Cushions,Dumbbells,Fiction,Foundation,Headphones,Jacket,Jeans,...,Non-fiction,Perfume,Resistance Bands,Shoes,Smartphone,Smartwatch,T-shirt,Treadmill,Wall Art,Yoga Mat
Category,,,,,,,,,,,,,,,,,,,,,
Beauty,0.0286,0.0280,0.0310,0.0310,0.0292,0.0277,0.1054,0.0280,0.0274,0.0283,...,0.0301,0.1009,0.0282,0.0269,0.0325,0.0268,0.0294,0.0293,0.0288,0.0256
Books,0.1093,0.1084,0.0286,0.0273,0.0280,0.1041,0.0280,0.0286,0.0279,0.0285,...,0.1053,0.0281,0.0307,0.0272,0.0267,0.0300,0.0311,0.0260,0.0256,0.0311
Electronics,0.0283,0.0292,0.0314,0.0273,0.0291,0.0275,0.0252,0.1096,0.0295,0.0256,...,0.0270,0.0297,0.0252,0.0282,0.1105,0.1054,0.0297,0.0287,0.0310,0.0288
Fashion,0.0262,0.0295,0.0326,0.0317,0.0295,0.0303,0.0281,0.0292,0.1054,0.1098,...,0.0271,0.0272,0.0277,0.1015,0.0282,0.0273,0.1127,0.0298,0.0263,0.0281
Fitness,0.0308,0.0266,0.0261,0.0286,0.1070,0.0290,0.0276,0.0285,0.0276,0.0295,...,0.0282,0.0264,0.1075,0.0274,0.0278,0.0281,0.0310,0.1070,0.0307,0.1066
Home Decor,0.0290,0.0279,0.1120,0.1078,0.0265,0.0256,0.0289,0.0271,0.0262,0.0301,...,0.0261,0.0279,0.0278,0.0267,0.0316,0.0293,0.0322,0.0304,0.1076,0.0295


In [30]:
def browsing_recommend(customer_id, n=5):

    customer_row = df_customer[
        df_customer["Customer_ID"] == customer_id
    ].iloc[0]

    browsing_categories = customer_row["Browsing_List"]
    purchased_products = set(customer_row["Purchase_List"])

    scores = pd.Series(
        0.0,
        index=category_product_probability.columns
    )

    for category in browsing_categories:

        if category in category_product_probability.index:
            scores += category_product_probability.loc[category]

    # Don't recommend products already purchased
    scores = scores.drop(
        labels=list(purchased_products),
        errors="ignore"
    )

    return scores.sort_values(
        ascending=False
    ).head(n)

In [31]:
customer_id = df_customer["Customer_ID"].iloc[0]

print("Customer:", customer_id)

print("\nBrowsing history:")
print(
    df_customer.loc[
        df_customer["Customer_ID"] == customer_id,
        "Browsing_List"
    ].iloc[0]
)

print("\nPurchase history:")
print(
    df_customer.loc[
        df_customer["Customer_ID"] == customer_id,
        "Purchase_List"
    ].iloc[0]
)

print("\nBrowsing-based recommendations:")
print(
    browsing_recommend(customer_id, 5)
)

Customer: C1000

Browsing history:
['Books', 'Fashion']

Purchase history:
['Biography', 'Jeans']

Browsing-based recommendations:
Product
T-shirt        0.143871
Comics         0.137929
Fiction        0.134386
Jacket         0.133256
Non-fiction    0.132342
dtype: float64


In [32]:
def get_cf_scores(customer_id):

    purchased_products = (
        user_item_matrix.loc[customer_id]
        [user_item_matrix.loc[customer_id] > 0]
        .index
        .tolist()
    )

    scores = pd.Series(
        0.0,
        index=item_similarity_df.index
    )

    for product in purchased_products:
        scores += item_similarity_df[product]

    # Remove products already purchased
    scores = scores.drop(
        labels=purchased_products,
        errors="ignore"
    )

    return scores

In [33]:
def get_browsing_scores(customer_id):

    customer_row = df_customer[
        df_customer["Customer_ID"] == customer_id
    ].iloc[0]

    browsing_categories = customer_row["Browsing_List"]
    purchased_products = set(customer_row["Purchase_List"])

    scores = pd.Series(
        0.0,
        index=category_product_probability.columns
    )

    for category in browsing_categories:

        if category in category_product_probability.index:
            scores += category_product_probability.loc[category]

    scores = scores.drop(
        labels=list(purchased_products),
        errors="ignore"
    )

    return scores

In [34]:
def min_max_normalize(scores):

    if scores.max() == scores.min():
        return scores * 0

    return (
        (scores - scores.min())
        / (scores.max() - scores.min())
    )

In [35]:
def hybrid_recommend(
    customer_id,
    n=5,
    cf_weight=0.7,
    browsing_weight=0.3
):

    cf_scores = get_cf_scores(customer_id)
    browsing_scores = get_browsing_scores(customer_id)

    # Normalize scores
    cf_scores = min_max_normalize(cf_scores)
    browsing_scores = min_max_normalize(browsing_scores)

    # Combine scores
    hybrid_scores = (
        cf_weight * cf_scores
        +
        browsing_weight * browsing_scores
    )

    return (
        hybrid_scores
        .sort_values(ascending=False)
        .head(n)
    )

In [36]:
customer_id = "C1000"

print("Customer:", customer_id)

print("\nPurchase History:")
print(
    df_customer.loc[
        df_customer["Customer_ID"] == customer_id,
        "Purchase_List"
    ].iloc[0]
)

print("\nBrowsing History:")
print(
    df_customer.loc[
        df_customer["Customer_ID"] == customer_id,
        "Browsing_List"
    ].iloc[0]
)

print("\nHybrid Recommendations:")
print(
    hybrid_recommend(customer_id, n=5)
)

Customer: C1000

Purchase History:
['Biography', 'Jeans']

Browsing History:
['Books', 'Fashion']

Hybrid Recommendations:
Product
Dumbbells           0.718190
Resistance Bands    0.685367
Lipstick            0.668331
Curtains            0.667467
Perfume             0.667139
dtype: float64


In [37]:
hybrid_recommend("C1000", n=5)

,0
Product,
Dumbbells,0.718190
Resistance Bands,0.685367
Lipstick,0.668331
Curtains,0.667467
Perfume,0.667139


In [62]:
# ============================================
# STEP 1: LEAKAGE-FREE TRAIN/TEST SPLIT
# ============================================

evaluation_data = []

for _, row in df_customer.iterrows():

    customer_id = row["Customer_ID"]
    purchases = row["Purchase_List"]

    # Only users with at least 2 purchases
    if len(purchases) >= 2:

        test_item = purchases[-1]
        train_items = purchases[:-1]

        evaluation_data.append({
            "Customer_ID": customer_id,
            "Train_Purchases": train_items,
            "Test_Item": test_item,
            "Browsing_Categories": row["Browsing_List"]
        })

evaluation_df = pd.DataFrame(evaluation_data)

print("Evaluation customers:", len(evaluation_df))
print(evaluation_df.head())

Evaluation customers: 6631
  Customer_ID                Train_Purchases Test_Item  \
0       C1000                    [Biography]     Jeans   
1       C1001  [Biography, Resistance Bands]   T-shirt   
2       C1004                        [Shoes]      Lamp   
3       C1010        [Biography, Smartphone]  Yoga Mat   
4       C1012          [Headphones, Fiction]  Yoga Mat   

             Browsing_Categories  
0               [Books, Fashion]  
1      [Books, Fitness, Fashion]  
2          [Fashion, Home Decor]  
3  [Books, Electronics, Fitness]  
4  [Electronics, Books, Fitness]  


In [63]:
# ============================================
# STEP 2: TRAINING USER-ITEM MATRIX
# ============================================

train_purchase_rows = []

for _, row in evaluation_df.iterrows():

    customer_id = row["Customer_ID"]

    for product in row["Train_Purchases"]:

        train_purchase_rows.append(
            (customer_id, product)
        )

train_purchase_df = pd.DataFrame(
    train_purchase_rows,
    columns=["Customer_ID", "Product"]
)

train_purchase_df["Purchase"] = 1

train_user_item_matrix = pd.crosstab(
    train_purchase_df["Customer_ID"],
    train_purchase_df["Product"]
)

print("Training interaction shape:",
      train_user_item_matrix.shape)

print("\nTraining interactions:",
      len(train_purchase_df))

Training interaction shape: (6631, 24)

Training interactions: 9967


In [64]:
# ============================================
# STEP 3: TRAINING-ONLY ITEM SIMILARITY
# ============================================

from sklearn.metrics.pairwise import cosine_similarity

train_item_similarity = cosine_similarity(
    train_user_item_matrix.T
)

train_item_similarity_df = pd.DataFrame(
    train_item_similarity,
    index=train_user_item_matrix.columns,
    columns=train_user_item_matrix.columns
)

print("Item similarity shape:",
      train_item_similarity_df.shape)

Item similarity shape: (24, 24)


In [65]:
# ============================================
# STEP 4: TRAINING-ONLY BROWSING MODEL
# ============================================

train_browsing_purchase_pairs = []

for _, row in evaluation_df.iterrows():

    browsing_categories = row["Browsing_Categories"]
    train_products = row["Train_Purchases"]

    for category in browsing_categories:

        for product in train_products:

            train_browsing_purchase_pairs.append(
                (category, product)
            )

train_browsing_purchase_df = pd.DataFrame(
    train_browsing_purchase_pairs,
    columns=["Category", "Product"]
)

train_category_product_counts = pd.crosstab(
    train_browsing_purchase_df["Category"],
    train_browsing_purchase_df["Product"]
)

train_category_product_probability = (
    train_category_product_counts
    .div(
        train_category_product_counts.sum(axis=1),
        axis=0
    )
)

print(
    "Training browsing-purchase pairs:",
    len(train_browsing_purchase_df)
)

print(
    "Probability matrix shape:",
    train_category_product_probability.shape
)

Training browsing-purchase pairs: 26606
Probability matrix shape: (6, 24)


In [66]:
# ============================================
# ITEM-CF SCORING
# ============================================

def get_train_cf_scores(customer_id):

    train_items = evaluation_df.loc[
        evaluation_df["Customer_ID"] == customer_id,
        "Train_Purchases"
    ].iloc[0]

    scores = pd.Series(
        0.0,
        index=train_item_similarity_df.index
    )

    for item in train_items:

        if item in train_item_similarity_df.columns:

            scores += train_item_similarity_df[item]

    # Remove products already purchased
    scores = scores.drop(
        labels=train_items,
        errors="ignore"
    )

    return scores

In [67]:
# ============================================
# BROWSING SCORING
# ============================================

def get_train_browsing_scores(customer_id):

    row = evaluation_df[
        evaluation_df["Customer_ID"] == customer_id
    ].iloc[0]

    browsing_categories = row["Browsing_Categories"]
    train_items = set(row["Train_Purchases"])

    scores = pd.Series(
        0.0,
        index=train_category_product_probability.columns
    )

    for category in browsing_categories:

        if category in train_category_product_probability.index:

            scores += train_category_product_probability.loc[
                category
            ]

    # Remove already purchased products
    scores = scores.drop(
        labels=list(train_items),
        errors="ignore"
    )

    return scores

In [68]:
# ============================================
# NORMALIZATION
# ============================================

def normalize_scores(scores):

    if len(scores) == 0:
        return scores

    if scores.max() == scores.min():
        return scores * 0

    return (
        (scores - scores.min()) /
        (scores.max() - scores.min())
    )

In [69]:
# ============================================
# HYBRID SCORING
# ============================================

def get_hybrid_scores(
    customer_id,
    cf_weight=0.7,
    browsing_weight=0.3
):

    cf_scores = get_train_cf_scores(customer_id)

    browsing_scores = get_train_browsing_scores(customer_id)

    # Normalize both models
    cf_scores = normalize_scores(cf_scores)
    browsing_scores = normalize_scores(browsing_scores)

    # Make sure both contain the same products
    all_products = sorted(
        set(cf_scores.index) |
        set(browsing_scores.index)
    )

    cf_scores = cf_scores.reindex(
        all_products,
        fill_value=0
    )

    browsing_scores = browsing_scores.reindex(
        all_products,
        fill_value=0
    )

    hybrid_scores = (
        cf_weight * cf_scores +
        browsing_weight * browsing_scores
    )

    return hybrid_scores

In [70]:
# ============================================
# EVALUATION FUNCTION
# ============================================

def evaluate_model(model_name, k=5,
                   cf_weight=0.7,
                   browsing_weight=0.3):

    hits = 0
    total = 0

    for _, row in evaluation_df.iterrows():

        customer_id = row["Customer_ID"]
        test_item = row["Test_Item"]

        # -------------------------
        # Generate recommendations
        # -------------------------

        if model_name == "CF":

            scores = get_train_cf_scores(customer_id)

        elif model_name == "Browsing":

            scores = get_train_browsing_scores(customer_id)

        elif model_name == "Hybrid":

            scores = get_hybrid_scores(
                customer_id,
                cf_weight,
                browsing_weight
            )

        else:
            raise ValueError("Unknown model")

        recommendations = (
            scores
            .sort_values(ascending=False)
            .head(k)
            .index
            .tolist()
        )

        # -------------------------
        # Check test item
        # -------------------------

        if test_item in recommendations:

            hits += 1

        total += 1

    hit_rate = hits / total

    precision = hits / (total * k)

    recall = hits / total

    return precision, recall, hit_rate

In [71]:
cf_precision, cf_recall, cf_hit = evaluate_model(
    "CF",
    k=5
)

print("Item-CF")
print("----------------------")
print(f"Precision@5: {cf_precision:.4f}")
print(f"Recall@5:    {cf_recall:.4f}")
print(f"Hit Rate@5: {cf_hit:.4f}")

Item-CF
----------------------
Precision@5: 0.0562
Recall@5:    0.2808
Hit Rate@5: 0.2808


In [72]:
browse_precision, browse_recall, browse_hit = evaluate_model(
    "Browsing",
    k=5
)

print("Browsing-Based")
print("----------------------")
print(f"Precision@5: {browse_precision:.4f}")
print(f"Recall@5:    {browse_recall:.4f}")
print(f"Hit Rate@5: {browse_hit:.4f}")

Browsing-Based
----------------------
Precision@5: 0.1221
Recall@5:    0.6103
Hit Rate@5: 0.6103


In [73]:
hybrid_precision, hybrid_recall, hybrid_hit = evaluate_model(
    "Hybrid",
    k=5,
    cf_weight=0.7,
    browsing_weight=0.3
)

print("Hybrid 70/30")
print("----------------------")
print(f"Precision@5: {hybrid_precision:.4f}")
print(f"Recall@5:    {hybrid_recall:.4f}")
print(f"Hit Rate@5: {hybrid_hit:.4f}")

Hybrid 70/30
----------------------
Precision@5: 0.1565
Recall@5:    0.7827
Hit Rate@5: 0.7827


In [74]:
# ============================================
# WEIGHT EXPERIMENT
# ============================================

weights = [
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.9, 0.1)
]

results = []

for cf_weight, browsing_weight in weights:

    precision, recall, hit_rate = evaluate_model(
        "Hybrid",
        k=5,
        cf_weight=cf_weight,
        browsing_weight=browsing_weight
    )

    results.append({
        "CF Weight": cf_weight,
        "Browsing Weight": browsing_weight,
        "Precision@5": precision,
        "Recall@5": recall,
        "Hit Rate@5": hit_rate
    })

results_df = pd.DataFrame(results)

results_df

,CF Weight,Browsing Weight,Precision@5,Recall@5,Hit Rate@5
0,0.5,0.5,0.195596,0.977982,0.977982
1,0.6,0.4,0.185734,0.928668,0.928668
2,0.7,0.3,0.156537,0.782687,0.782687
3,0.8,0.2,0.116905,0.584527,0.584527
4,0.9,0.1,0.082612,0.413060,0.413060


In [75]:
# ============================================
# CHECK TEST ITEM RECOMMENDATION DISTRIBUTION
# ============================================

cf_hits = 0
browse_hits = 0
hybrid_hits = 0
total = 0

for _, row in evaluation_df.iterrows():

    customer_id = row["Customer_ID"]
    test_item = row["Test_Item"]

    # CF
    cf_scores = get_train_cf_scores(customer_id)
    cf_recs = (
        cf_scores
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    # Browsing
    browsing_scores = get_train_browsing_scores(customer_id)
    browsing_recs = (
        browsing_scores
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    # Hybrid 50/50
    hybrid_scores = get_hybrid_scores(
        customer_id,
        cf_weight=0.5,
        browsing_weight=0.5
    )

    hybrid_recs = (
        hybrid_scores
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    if test_item in cf_recs:
        cf_hits += 1

    if test_item in browsing_recs:
        browse_hits += 1

    if test_item in hybrid_recs:
        hybrid_hits += 1

    total += 1

print("Total evaluation customers:", total)
print("CF hits:", cf_hits)
print("Browsing hits:", browse_hits)
print("Hybrid 50/50 hits:", hybrid_hits)

print("\nHit Rates:")
print("CF:", cf_hits / total)
print("Browsing:", browse_hits / total)
print("Hybrid:", hybrid_hits / total)

Total evaluation customers: 6631
CF hits: 1862
Browsing hits: 4047
Hybrid 50/50 hits: 6485

Hit Rates:
CF: 0.2808022922636103
Browsing: 0.6103151862464183
Hybrid: 0.9779822047956568


In [76]:
# ============================================
# CHECK TEST ITEM FREQUENCY
# ============================================

test_item_counts = evaluation_df["Test_Item"].value_counts()

print(test_item_counts)

Test_Item
T-shirt             322
Curtains            305
Foundation          292
Yoga Mat            289
Fiction             289
Moisturizer         286
Lipstick            281
Treadmill           278
Non-fiction         276
Smartphone          273
Biography           273
Cushions            273
Lamp                272
Dumbbells           272
Jacket              272
Perfume             268
Comics              267
Jeans               266
Wall Art            266
Laptop              265
Shoes               264
Headphones          261
Resistance Bands    261
Smartwatch          260
Name: count, dtype: int64


In [77]:
# ============================================
# STEP 12: HYBRID CONTRIBUTION ANALYSIS
# ============================================

test_weights = [
    (0.0, 1.0),
    (0.1, 0.9),
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.9, 0.1),
    (1.0, 0.0)
]

weight_results = []

for cf_weight, browsing_weight in test_weights:

    _, _, hit_rate = evaluate_model(
        "Hybrid",
        k=5,
        cf_weight=cf_weight,
        browsing_weight=browsing_weight
    )

    weight_results.append({
        "CF Weight": cf_weight,
        "Browsing Weight": browsing_weight,
        "Hit Rate@5": hit_rate
    })

weight_results_df = pd.DataFrame(weight_results)

print(weight_results_df)

    CF Weight  Browsing Weight  Hit Rate@5
0         0.0              1.0    0.610315
1         0.1              0.9    0.743478
2         0.2              0.8    0.876037
3         0.3              0.7    0.940431
4         0.4              0.6    0.970442
5         0.5              0.5    0.977982
6         0.6              0.4    0.928668
7         0.7              0.3    0.782687
8         0.8              0.2    0.584527
9         0.9              0.1    0.413060
10        1.0              0.0    0.280802


In [78]:
# ============================================
# STEP 13: CHECK SCORE SEPARATION
# ============================================

customer_id = evaluation_df.iloc[0]["Customer_ID"]

cf_scores = get_train_cf_scores(customer_id)
browse_scores = get_train_browsing_scores(customer_id)

cf_norm = normalize_scores(cf_scores)
browse_norm = normalize_scores(browse_scores)

score_comparison = pd.DataFrame({
    "CF": cf_norm,
    "Browsing": browse_norm
}).fillna(0)

score_comparison["Hybrid_50_50"] = (
    0.5 * score_comparison["CF"] +
    0.5 * score_comparison["Browsing"]
)

print(score_comparison.sort_values(
    "Hybrid_50_50",
    ascending=False
).head(10))

                  CF  Browsing  Hybrid_50_50
Product                                     
Jeans       0.827430  0.980802      0.904116
Jacket      0.673944  0.887732      0.780838
T-shirt     0.614049  0.932741      0.773395
Shoes       0.576580  0.794873      0.685727
Dumbbells   0.996439  0.057673      0.527056
Curtains    1.000000  0.045148      0.522574
Comics      0.000000  1.000000      0.500000
Smartwatch  0.880523  0.074475      0.477499
Laptop      0.885724  0.045218      0.465471
Lamp        0.868225  0.026169      0.447197


In [79]:
# ============================================
# STEP 14: CATEGORY-BASED ANALYSIS
# ============================================

category_top_products = {}

for category in train_category_product_probability.index:

    top_products = (
        train_category_product_probability
        .loc[category]
        .sort_values(ascending=False)
        .head(5)
    )

    category_top_products[category] = top_products.index.tolist()

for category, products in category_top_products.items():

    print(f"\n{category}")
    print("-" * 30)

    for i, product in enumerate(products, 1):
        print(f"{i}. {product}")


Beauty
------------------------------
1. Moisturizer
2. Lipstick
3. Perfume
4. Foundation
5. Smartphone

Books
------------------------------
1. Comics
2. Biography
3. Non-fiction
4. Fiction
5. Smartwatch

Electronics
------------------------------
1. Smartphone
2. Smartwatch
3. Headphones
4. Laptop
5. Wall Art

Fashion
------------------------------
1. Jeans
2. T-shirt
3. Jacket
4. Shoes
5. Cushions

Fitness
------------------------------
1. Resistance Bands
2. Dumbbells
3. Treadmill
4. Yoga Mat
5. Biography

Home Decor
------------------------------
1. Cushions
2. Wall Art
3. Curtains
4. Lamp
5. Lipstick


In [80]:
# ============================================
# STEP 15: CHECK CATEGORY-PRODUCT COVERAGE
# ============================================

print("Number of browsing categories:",
      len(train_category_product_probability.index))

print("Number of products:",
      len(train_category_product_probability.columns))

print("\nCategories:")
print(
    train_category_product_probability.index.tolist()
)

print("\nProducts:")
print(
    train_category_product_probability.columns.tolist()
)

Number of browsing categories: 6
Number of products: 24

Categories:
['Beauty', 'Books', 'Electronics', 'Fashion', 'Fitness', 'Home Decor']

Products:
['Biography', 'Comics', 'Curtains', 'Cushions', 'Dumbbells', 'Fiction', 'Foundation', 'Headphones', 'Jacket', 'Jeans', 'Lamp', 'Laptop', 'Lipstick', 'Moisturizer', 'Non-fiction', 'Perfume', 'Resistance Bands', 'Shoes', 'Smartphone', 'Smartwatch', 'T-shirt', 'Treadmill', 'Wall Art', 'Yoga Mat']


In [81]:
# ============================================
# RESTORE LIST COLUMNS
# ============================================

import ast

df_customer["Browsing_List"] = (
    df_customer["Browsing_History"]
    .apply(ast.literal_eval)
)

df_customer["Purchase_List"] = (
    df_customer["Purchase_History"]
    .apply(ast.literal_eval)
)

print("Columns restored successfully.")

print(
    df_customer[
        ["Customer_ID", "Browsing_List", "Purchase_List"]
    ].head()
)

Columns restored successfully.
  Customer_ID              Browsing_List  \
0       C1000           [Books, Fashion]   
1       C1001  [Books, Fitness, Fashion]   
2       C1002              [Electronics]   
3       C1003               [Home Decor]   
4       C1004      [Fashion, Home Decor]   

                            Purchase_List  
0                      [Biography, Jeans]  
1  [Biography, Resistance Bands, T-shirt]  
2                            [Smartphone]  
3                              [Wall Art]  
4                           [Shoes, Lamp]  


In [82]:
print(df_customer.columns.tolist())

['Customer_ID', 'Age', 'Gender', 'Location', 'Browsing_History', 'Purchase_History', 'Customer_Segment', 'Avg_Order_Value', 'Holiday', 'Season', 'Unnamed: 10', 'Browsing_List', 'Purchase_List']


In [83]:
# ============================================
# STEP 16: FINAL MODEL TRAINING
# ============================================

print("Building final recommendation model...")
print()

# --------------------------------------------
# 1. Final User-Item Matrix
# --------------------------------------------

final_user_item_matrix = pd.crosstab(
    purchase_interactions["Customer_ID"],
    purchase_interactions["Product"]
)

# --------------------------------------------
# 2. Final Item Similarity
# --------------------------------------------

final_item_similarity = cosine_similarity(
    final_user_item_matrix.T
)

final_item_similarity_df = pd.DataFrame(
    final_item_similarity,
    index=final_user_item_matrix.columns,
    columns=final_user_item_matrix.columns
)

# --------------------------------------------
# 3. Final Browsing-Purchase Relationships
# --------------------------------------------

final_browsing_purchase_pairs = []

for _, row in df_customer.iterrows():

    browsing_categories = row["Browsing_List"]
    purchased_products = row["Purchase_List"]

    for category in browsing_categories:

        for product in purchased_products:

            final_browsing_purchase_pairs.append(
                (category, product)
            )

final_browsing_purchase_df = pd.DataFrame(
    final_browsing_purchase_pairs,
    columns=["Category", "Product"]
)

final_category_product_counts = pd.crosstab(
    final_browsing_purchase_df["Category"],
    final_browsing_purchase_df["Product"]
)

final_category_product_probability = (
    final_category_product_counts
    .div(
        final_category_product_counts.sum(axis=1),
        axis=0
    )
)

# --------------------------------------------
# 4. Product Popularity
# --------------------------------------------

final_product_popularity = (
    purchase_interactions["Product"]
    .value_counts()
)

print("Final User-Item Matrix:",
      final_user_item_matrix.shape)

print("Final Item Similarity:",
      final_item_similarity_df.shape)

print("Final Category-Product Matrix:",
      final_category_product_probability.shape)

print("Final Products:",
      len(final_product_popularity))

print("\nFinal model training completed.")

Building final recommendation model...

Final User-Item Matrix: (10000, 24)
Final Item Similarity: (24, 24)
Final Category-Product Matrix: (6, 24)
Final Products: 24

Final model training completed.


In [87]:
# ============================================
# STEP 17: FINAL RECOMMENDATION FUNCTION
# ============================================

FINAL_CF_WEIGHT = 0.5
FINAL_BROWSING_WEIGHT = 0.5


def final_recommend_products(
    customer_id,
    n=5
):

    # ----------------------------------------
    # Check whether customer exists
    # ----------------------------------------

    if customer_id not in final_user_item_matrix.index:

        # Cold-start fallback
        return final_product_popularity.head(n)


    # ----------------------------------------
    # Customer's purchased products
    # ----------------------------------------

    purchased_products = (
        final_user_item_matrix.loc[customer_id]
        [
            final_user_item_matrix.loc[customer_id] > 0
        ]
        .index
        .tolist()
    )


    # ----------------------------------------
    # Item-CF scores
    # ----------------------------------------

    cf_scores = pd.Series(
        0.0,
        index=final_item_similarity_df.index
    )

    for product in purchased_products:

        if product in final_item_similarity_df.columns:

            cf_scores += final_item_similarity_df[product]


    # Remove already purchased products
    cf_scores = cf_scores.drop(
        labels=purchased_products,
        errors="ignore"
    )


    # ----------------------------------------
    # Browsing scores
    # ----------------------------------------

    customer_row = df_customer[
        df_customer["Customer_ID"] == customer_id
    ]

    browsing_scores = pd.Series(
        0.0,
        index=final_category_product_probability.columns
    )

    if len(customer_row) > 0:

        browsing_categories = (
            customer_row.iloc[0]["Browsing_List"]
        )

        for category in browsing_categories:

            if category in final_category_product_probability.index:

                browsing_scores += (
                    final_category_product_probability
                    .loc[category]
                )


    # Remove already purchased products
    browsing_scores = browsing_scores.drop(
        labels=purchased_products,
        errors="ignore"
    )


    # ----------------------------------------
    # Normalize
    # ----------------------------------------

    cf_scores = normalize_scores(cf_scores)
    browsing_scores = normalize_scores(browsing_scores)


    # ----------------------------------------
    # Align products
    # ----------------------------------------

    all_products = sorted(
        set(cf_scores.index) |
        set(browsing_scores.index)
    )

    cf_scores = cf_scores.reindex(
        all_products,
        fill_value=0
    )

    browsing_scores = browsing_scores.reindex(
        all_products,
        fill_value=0
    )


    # ----------------------------------------
    # Hybrid score
    # ----------------------------------------

    hybrid_scores = (
        FINAL_CF_WEIGHT * cf_scores
        +
        FINAL_BROWSING_WEIGHT * browsing_scores
    )


    # ----------------------------------------
    # Final recommendations
    # ----------------------------------------

    recommendations = (
        hybrid_scores
        .sort_values(ascending=False)
        .head(n)
    )

    return recommendations

In [88]:
# ============================================
# STEP 18: TEST CUSTOMER
# ============================================

customer_id = "C1000"

print("Customer:", customer_id)

customer_info = df_customer[
    df_customer["Customer_ID"] == customer_id
].iloc[0]

print("\nBrowsing:")
print(customer_info["Browsing_List"])

print("\nPurchased:")
print(customer_info["Purchase_List"])

print("\nRecommendations:")
print(
    final_recommend_products(
        customer_id,
        n=5
    )
)

Customer: C1000

Browsing:
['Books', 'Fashion']

Purchased:
['Biography', 'Jeans']

Recommendations:
Product
Fiction             0.628635
T-shirt             0.579752
Dumbbells           0.530317
Non-fiction         0.521880
Resistance Bands    0.509801
dtype: float64


In [89]:
# ============================================
# STEP 19: COLD-START TEST
# ============================================

new_customer_id = "NEW_CUSTOMER_001"

print(
    final_recommend_products(
        new_customer_id,
        n=5
    )
)

Product
Moisturizer    882
Curtains       876
T-shirt        875
Smartphone     863
Headphones     856
Name: count, dtype: int64


In [90]:
# ============================================
# STEP 20: NEW CUSTOMER WITH BROWSING
# ============================================

def cold_start_browsing_recommend(
    browsing_categories,
    n=5
):

    scores = pd.Series(
        0.0,
        index=final_category_product_probability.columns
    )

    for category in browsing_categories:

        if category in final_category_product_probability.index:

            scores += (
                final_category_product_probability.loc[category]
            )

    return (
        scores
        .sort_values(ascending=False)
        .head(n)
    )


# Test new customer browsing Electronics
new_customer_browsing = ["Electronics"]

print("New Customer Browsing:")
print(new_customer_browsing)

print("\nRecommendations:")
print(
    cold_start_browsing_recommend(
        new_customer_browsing,
        n=5
    )
)

New Customer Browsing:
['Electronics']

Recommendations:
Product
Smartphone     0.110542
Headphones     0.109645
Laptop         0.105546
Smartwatch     0.105418
Moisturizer    0.031894
dtype: float64


In [91]:
print(
    cold_start_browsing_recommend(
        ["Books", "Fashion"],
        n=5
    )
)

Product
T-shirt      0.143871
Jeans        0.138291
Comics       0.137929
Biography    0.135495
Fiction      0.134386
dtype: float64


In [92]:
import joblib

# Save trained recommendation components

joblib.dump(
    final_item_similarity_df,
    "item_similarity.pkl"
)

joblib.dump(
    final_category_product_probability,
    "category_product_probability.pkl"
)

joblib.dump(
    final_product_popularity,
    "product_popularity.pkl"
)

joblib.dump(
    list(final_user_item_matrix.columns),
    "product_list.pkl"
)

# Save model configuration
model_config = {
    "cf_weight": FINAL_CF_WEIGHT,
    "browsing_weight": FINAL_BROWSING_WEIGHT,
    "recommendation_count": 5
}

joblib.dump(
    model_config,
    "recommendation_config.pkl"
)

print("All recommendation model artifacts saved successfully.")

All recommendation model artifacts saved successfully.


In [93]:
import os

files = [
    "item_similarity.pkl",
    "category_product_probability.pkl",
    "product_popularity.pkl",
    "product_list.pkl",
    "recommendation_config.pkl"
]

for file in files:
    print(file, "->", os.path.getsize(file), "bytes")

item_similarity.pkl -> 5706 bytes
category_product_probability.pkl -> 2530 bytes
product_popularity.pkl -> 2063 bytes
product_list.pkl -> 286 bytes
recommendation_config.pkl -> 89 bytes


In [94]:
def recommend_for_website(
    customer_id,
    browsing_categories=None,
    purchase_history=None,
    n=5
):

    browsing_categories = browsing_categories or []
    purchase_history = purchase_history or []

    # ------------------------------------------------
    # CASE 1: Existing customer with purchase history
    # ------------------------------------------------

    if customer_id in final_user_item_matrix.index:

        recommendations = final_recommend_products(
            customer_id,
            n=n
        )

        return [
            {
                "product": product,
                "score": float(score)
            }
            for product, score in recommendations.items()
        ]

    # ------------------------------------------------
    # CASE 2: New customer with browsing history
    # ------------------------------------------------

    if len(browsing_categories) > 0:

        recommendations = cold_start_browsing_recommend(
            browsing_categories,
            n=n
        )

        return [
            {
                "product": product,
                "score": float(score)
            }
            for product, score in recommendations.items()
        ]

    # ------------------------------------------------
    # CASE 3: Completely new customer
    # ------------------------------------------------

    recommendations = final_product_popularity.head(n)

    return [
        {
            "product": product,
            "score": float(score)
        }
        for product, score in recommendations.items()
    ]

In [95]:
result = recommend_for_website(
    customer_id="C1000",
    browsing_categories=["Books", "Fashion"],
    purchase_history=["Biography", "Jeans"],
    n=5
)

print(result)

[{'product': 'Fiction', 'score': 0.6286354938968279}, {'product': 'T-shirt', 'score': 0.5797524851571842}, {'product': 'Dumbbells', 'score': 0.5303172698329383}, {'product': 'Non-fiction', 'score': 0.5218801557409521}, {'product': 'Resistance Bands', 'score': 0.509800873030973}]


In [96]:
result = recommend_for_website(
    customer_id="NEW_CUSTOMER_001",
    browsing_categories=["Electronics"],
    purchase_history=[],
    n=5
)

print(result)

[{'product': 'Smartphone', 'score': 0.11054182144229538}, {'product': 'Headphones', 'score': 0.10964519021391059}, {'product': 'Laptop', 'score': 0.1055463045984373}, {'product': 'Smartwatch', 'score': 0.10541821442295377}, {'product': 'Moisturizer', 'score': 0.031894453695401566}]


In [97]:
result = recommend_for_website(
    customer_id="NEW_CUSTOMER_002",
    browsing_categories=[],
    purchase_history=[],
    n=5
)

print(result)

[{'product': 'Moisturizer', 'score': 882.0}, {'product': 'Curtains', 'score': 876.0}, {'product': 'T-shirt', 'score': 875.0}, {'product': 'Smartphone', 'score': 863.0}, {'product': 'Headphones', 'score': 856.0}]


In [99]:
import joblib

joblib.dump(
    final_item_similarity_df,
    "item_similarity.pkl"
)

joblib.dump(
    final_category_product_probability,
    "category_product_probability.pkl"
)

joblib.dump(
    final_product_popularity,
    "product_popularity.pkl"
)

joblib.dump(
    list(final_user_item_matrix.columns),
    "product_list.pkl"
)

model_config = {
    "cf_weight": FINAL_CF_WEIGHT,
    "browsing_weight": FINAL_BROWSING_WEIGHT,
    "recommendation_count": 5
}

joblib.dump(
    model_config,
    "recommendation_config.pkl"
)

print("All recommendation model artifacts saved successfully.")

All recommendation model artifacts saved successfully.


In [100]:
import os

files = [
    "item_similarity.pkl",
    "category_product_probability.pkl",
    "product_popularity.pkl",
    "product_list.pkl",
    "recommendation_config.pkl"
]

for file in files:
    print(file, "->", os.path.exists(file))

item_similarity.pkl -> True
category_product_probability.pkl -> True
product_popularity.pkl -> True
product_list.pkl -> True
recommendation_config.pkl -> True


In [101]:
import joblib

item_sim = joblib.load("item_similarity.pkl")
category_prob = joblib.load("category_product_probability.pkl")
popularity = joblib.load("product_popularity.pkl")
products = joblib.load("product_list.pkl")
config = joblib.load("recommendation_config.pkl")

print("Item similarity:", item_sim.shape)
print("Category-product:", category_prob.shape)
print("Number of products:", len(products))
print("Config:", config)

Item similarity: (24, 24)
Category-product: (6, 24)
Number of products: 24
Config: {'cf_weight': 0.5, 'browsing_weight': 0.5, 'recommendation_count': 5}


In [103]:
result = recommend_for_website(
    customer_id="C1000",
    browsing_categories=["Books", "Fashion"],
    purchase_history=["Biography", "Jeans"],
    n=5
)

print(result)

[{'product': 'Fiction', 'score': 0.6286354938968279}, {'product': 'T-shirt', 'score': 0.5797524851571842}, {'product': 'Dumbbells', 'score': 0.5303172698329383}, {'product': 'Non-fiction', 'score': 0.5218801557409521}, {'product': 'Resistance Bands', 'score': 0.509800873030973}]


In [104]:
result = recommend_for_website(
    customer_id="NEW_CUSTOMER_001",
    browsing_categories=["Electronics"],
    purchase_history=[],
    n=5
)

print(result)

[{'product': 'Smartphone', 'score': 0.11054182144229538}, {'product': 'Headphones', 'score': 0.10964519021391059}, {'product': 'Laptop', 'score': 0.1055463045984373}, {'product': 'Smartwatch', 'score': 0.10541821442295377}, {'product': 'Moisturizer', 'score': 0.031894453695401566}]


In [105]:
result = recommend_for_website(
    customer_id="NEW_CUSTOMER_002",
    browsing_categories=[],
    purchase_history=[],
    n=5
)

print(result)

[{'product': 'Moisturizer', 'score': 882.0}, {'product': 'Curtains', 'score': 876.0}, {'product': 'T-shirt', 'score': 875.0}, {'product': 'Smartphone', 'score': 863.0}, {'product': 'Headphones', 'score': 856.0}]


In [106]:
recommend_for_website(...)

[{'product': 'Moisturizer', 'score': 882.0},
 {'product': 'Curtains', 'score': 876.0},
 {'product': 'T-shirt', 'score': 875.0},
 {'product': 'Smartphone', 'score': 863.0},
 {'product': 'Headphones', 'score': 856.0}]

In [107]:
%%writefile recommendation_model.py

import joblib
import pandas as pd


# ============================================================
# LOAD MODEL ARTIFACTS
# ============================================================

item_similarity_df = joblib.load(
    "item_similarity.pkl"
)

category_product_probability = joblib.load(
    "category_product_probability.pkl"
)

product_popularity = joblib.load(
    "product_popularity.pkl"
)

product_list = joblib.load(
    "product_list.pkl"
)

model_config = joblib.load(
    "recommendation_config.pkl"
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

CF_WEIGHT = model_config["cf_weight"]
BROWSING_WEIGHT = model_config["browsing_weight"]
DEFAULT_N = model_config["recommendation_count"]


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_scores(scores):

    if len(scores) == 0:
        return scores

    if scores.max() == scores.min():
        return scores * 0

    return (
        (scores - scores.min())
        / (scores.max() - scores.min())
    )


# ============================================================
# BROWSING-BASED COLD START
# ============================================================

def cold_start_browsing_recommend(
    browsing_categories,
    n=5
):

    scores = pd.Series(
        0.0,
        index=category_product_probability.columns
    )

    for category in browsing_categories:

        if category in category_product_probability.index:

            scores += (
                category_product_probability
                .loc[category]
            )

    return (
        scores
        .sort_values(ascending=False)
        .head(n)
    )


# ============================================================
# WEBSITE RECOMMENDATION FUNCTION
# ============================================================

def recommend_for_website(
    customer_id,
    browsing_categories=None,
    purchase_history=None,
    n=None
):

    browsing_categories = browsing_categories or []
    purchase_history = purchase_history or []

    if n is None:
        n = DEFAULT_N


    # --------------------------------------------------------
    # CASE 1: EXISTING CUSTOMER
    # --------------------------------------------------------

    if customer_id in item_similarity_df.index:

        purchased_products = [
            product
            for product in purchase_history
            if product in item_similarity_df.index
        ]

        # If website provides purchase history,
        # use that history directly.
        if len(purchased_products) > 0:

            cf_scores = pd.Series(
                0.0,
                index=item_similarity_df.index
            )

            for product in purchased_products:

                cf_scores += (
                    item_similarity_df[product]
                )

            cf_scores = cf_scores.drop(
                labels=purchased_products,
                errors="ignore"
            )

            browsing_scores = pd.Series(
                0.0,
                index=category_product_probability.columns
            )

            for category in browsing_categories:

                if category in category_product_probability.index:

                    browsing_scores += (
                        category_product_probability
                        .loc[category]
                    )

            browsing_scores = browsing_scores.drop(
                labels=purchased_products,
                errors="ignore"
            )

            cf_scores = normalize_scores(cf_scores)
            browsing_scores = normalize_scores(browsing_scores)

            all_products = sorted(
                set(cf_scores.index)
                |
                set(browsing_scores.index)
            )

            cf_scores = cf_scores.reindex(
                all_products,
                fill_value=0
            )

            browsing_scores = browsing_scores.reindex(
                all_products,
                fill_value=0
            )

            hybrid_scores = (
                CF_WEIGHT * cf_scores
                +
                BROWSING_WEIGHT * browsing_scores
            )

            recommendations = (
                hybrid_scores
                .sort_values(ascending=False)
                .head(n)
            )

            return [
                {
                    "product": product,
                    "score": float(score)
                }
                for product, score in recommendations.items()
            ]


    # --------------------------------------------------------
    # CASE 2: NEW CUSTOMER WITH BROWSING HISTORY
    # --------------------------------------------------------

    if len(browsing_categories) > 0:

        recommendations = cold_start_browsing_recommend(
            browsing_categories,
            n=n
        )

        return [
            {
                "product": product,
                "score": float(score)
            }
            for product, score in recommendations.items()
        ]


    # --------------------------------------------------------
    # CASE 3: COMPLETELY NEW CUSTOMER
    # --------------------------------------------------------

    recommendations = (
        product_popularity
        .head(n)
    )

    return [
        {
            "product": product,
            "score": float(score)
        }
        for product, score in recommendations.items()
    ]

Writing recommendation_model.py


In [108]:
import importlib
import recommendation_model

importlib.reload(recommendation_model)

result = recommendation_model.recommend_for_website(
    customer_id="NEW_CUSTOMER_001",
    browsing_categories=["Electronics"],
    purchase_history=[],
    n=5
)

print(result)

[{'product': 'Smartphone', 'score': 0.11054182144229538}, {'product': 'Headphones', 'score': 0.10964519021391059}, {'product': 'Laptop', 'score': 0.1055463045984373}, {'product': 'Smartwatch', 'score': 0.10541821442295377}, {'product': 'Moisturizer', 'score': 0.031894453695401566}]


In [109]:
result = recommendation_model.recommend_for_website(
    customer_id="NEW_CUSTOMER_003",
    browsing_categories=["Electronics"],
    purchase_history=["iPhone 20 Ultra"],
    n=5
)

print(result)

[{'product': 'Smartphone', 'score': 0.11054182144229538}, {'product': 'Headphones', 'score': 0.10964519021391059}, {'product': 'Laptop', 'score': 0.1055463045984373}, {'product': 'Smartwatch', 'score': 0.10541821442295377}, {'product': 'Moisturizer', 'score': 0.031894453695401566}]


In [110]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles:")
for file in os.listdir():
    print(file)

Current folder:
/content

Files:
.config
product_list.pkl
item_similarity.pkl
recommendation_model.py
recommendation_config.pkl
__pycache__
category_product_probability.pkl
customer_data_collection (1).csv
customer_data_collection.csv
product_popularity.pkl
sample_data
